# MobileFaceNet

## Add New Embeddings

In [2]:
!pip3 install torch torchvision torchaudio

In [24]:
import numpy as np

In [3]:
import os

# cd to /content/drive/MyDrive/Face_Dataset
%cd Face_Dataset

print(os.getcwd())

g:\.thesis\named-ai\data-preprocessing\Face_Dataset
g:\.thesis\named-ai\data-preprocessing\Face_Dataset


In [ ]:
# Step 1 Clone the MobileFaceNet repository
!git clone https://github.com/foamliu/MobileFaceNet.git

Cloning into 'MobileFaceNet'...
remote: Enumerating objects: 589, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 589 (delta 186), reused 184 (delta 184), pack-reused 383 (from 1)
Receiving objects: 100% (589/589), 5.95 MiB | 9.34 MiB/s, done.
Resolving deltas: 100% (224/224), done.
Updating files: 100% (236/236), done.


In [11]:
%cd ..

g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet


In [ ]:
print("Current working directory:", os.getcwd())

# Step 2: Change directory into the repo
%cd ..

# Step 3: Make a directory for weights
!mkdir -p weights

# Step 4: Change into weights directory
%cd weights

# Step 5: Download the pretrained model
#!wget https://github.com/foamliu/MobileFaceNet/releases/download/v1.0/mobilefacenet.pt

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet\weights\weights\weights
[WinError 2] The system cannot find the file specified: 'MobileFaceNet'
g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet\weights\weights\weights
g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet\weights\weights\weights\weights


In [7]:
%cd MobileFaceNet

[Errno 2] No such file or directory: 'MobileFaceNet'
/Users/kyle/repos/named-ai/data-preprocessing/Face_Dataset/MobileFaceNet/weights


In [12]:
from mobilefacenet import MobileFaceNet
import torch
import time

# Load model
filename = 'weights/mobilefacenet.pt'
print(f'Loading {filename}...')
start = time.time()
model = MobileFaceNet()
model.load_state_dict(torch.load(filename, map_location=torch.device('cpu')))
model.eval()
print('Elapsed {:.2f} sec'.format(time.time() - start))


Loading weights/mobilefacenet.pt...
Elapsed 0.16 sec


In [13]:
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

def get_embedding(image_path):
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0)  # Shape: (1, 3, 112, 112)
    with torch.no_grad():
        embedding = model(input_tensor)
        return embedding.squeeze()  # Shape: (128,)


In [ ]:
#unsure how to implement this into this model
import cv2, torch, os

def build_facebank(facebank_dir="facebank"):
    names, embeddings = [], []
    #for folder structure images/PersonName/Person Name_x.jpg
    for person_name in os.listdir(facebank_dir):
        person_dir = os.path.join(facebank_dir, person_name)
        if not os.path.isdir(person_dir):
            continue

        person_embeddings = []
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue
            faces = app.get(img) # change to how ur model handles getting images from file
            if len(faces) == 0:
                continue

            emb = torch.tensor(faces[0]["embedding"])
            person_embeddings.append(emb)

        if person_embeddings:
            # Average embedding for that person
            mean_emb = torch.stack(person_embeddings).mean(dim=0)
            mean_emb = torch.nn.functional.normalize(mean_emb, p=2, dim=0)

            names.append(person_name)
            embeddings.append(mean_emb)

    if not embeddings:
        raise ValueError("No embeddings found in facebank.")

    facebank = torch.stack(embeddings)
    print(f"Built facebank with {len(names)} identities.")
    return names, facebank



def save_facebank(names, embeddings, path="facebank.pt"):
    torch.save({"names": names, "embeddings": embeddings}, path)
    print(f"Facebank saved to {path}")


def load_facebank(path="facebank.pt"):
    data = torch.load(path)
    print(f"Loaded facebank with {len(data['names'])} identities.")
    return data["names"], data["embeddings"]

In [14]:
face_db = {}
face_db['kyle'] = get_embedding('my_images/kyle_183.jpg')
face_db['Tom Cruise'] = get_embedding('my_images/Tom Cruise_12.jpg')

face_db.keys()

dict_keys(['kyle', 'Tom Cruise'])

In [15]:
# test_embedding = get_embedding('my_images/kyle_156.jpg')
# test_embedding = get_embedding('../processed_whole_face_dataset/val/kyle/kyle_161.jpg')
# test_embedding = get_embedding('../processed_whole_face_dataset/val/Zac Efron/Zac Efron_2.jpg')

test_embedding = get_embedding('../processed_whole_face_dataset/val/Tom Cruise/Tom Cruise_17.jpg')
print(test_embedding)

tensor([-8.9243e-02,  1.2614e+00, -4.9276e-01,  1.9606e+00,  1.0549e+00,
         1.3994e+00, -2.0412e+00,  1.1254e+00,  1.2575e+00,  6.8627e-01,
         8.5237e-02,  2.1320e+00,  5.3142e-01,  1.9682e+00, -2.5576e+00,
         8.5788e-01,  6.0927e-01, -4.9489e-01,  1.3770e+00,  1.0110e+00,
        -1.0470e+00,  4.2657e+00,  2.0887e+00,  1.6598e+00, -2.3915e-01,
         2.7591e+00, -1.1048e-03, -1.6891e+00,  6.5124e-01, -8.4227e-01,
         1.8632e+00,  2.4158e+00, -1.3980e+00, -2.6148e-01, -1.1415e+00,
         1.4403e+00,  5.3336e-02,  1.2166e+00, -4.3740e-01,  1.5985e+00,
         3.4416e-01,  1.5826e+00, -1.8154e+00,  3.4685e-01,  7.0870e-01,
        -6.9588e-01, -9.5989e-01, -6.4239e-01,  1.7788e+00, -4.6110e+00,
         1.1667e+00, -1.9093e+00,  1.8574e+00, -3.2795e+00, -3.5931e+00,
        -2.0152e-01,  2.9221e-01, -1.1172e+00,  2.0644e+00, -4.6767e-01,
         1.4042e+00,  1.3651e+00,  2.7450e-01, -1.6604e+00,  1.4132e-01,
        -6.9465e-01,  1.0509e+00,  4.1445e-02,  3.4

In [56]:
from torch.nn.functional import cosine_similarity

def recognize_face(test_embedding, face_db, threshold=0.6):
    max_sim = 0
    identity = "Unknown"
    for name, db_embedding in face_db.items():
        sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
        if sim.item() > max_sim and sim.item() > threshold:
            max_sim = sim.item()
            identity = name
    return identity, max_sim


In [ ]:
def recognize_face(image, face_db, threshold=0.5):
    """
    Args:
        image: input image for recognition
        face_db: dict {name: embedding}
        threshold: minimum similarity to accept a match

    Returns:
        (best_match_name, best_similarity)
    """
    if not face_db:
        print("[WARN] face_db is empty.")
        return "Unknown", 0.0

    target_emb = get_embedding(image)
    best_match = "Unknown"
    best_score = -1

    for name, db_emb in face_db.items():
        # Compute cosine similarity
        sim = np.dot(target_emb, db_emb) / (np.linalg.norm(target_emb) * np.linalg.norm(db_emb))
        
        if sim > best_score:
            best_score = sim
            best_match = name

    if best_score < threshold:
        best_match = "Unknown"

    return best_match, best_score

In [21]:
print("Current working directory:", os.getcwd())
%cd ..

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset
g:\.thesis\named-ai\data-preprocessing


In [36]:
# recognize face

name, score = recognize_face('my_images/kyle_156.jpg', face_db)
print(f"Recognized: {name} (Similarity: {score:.2f})")

Recognized: kyle (Similarity: 0.71)


C:\Users\julia\AppData\Local\Temp\ipykernel_13284\3388533542.py:24: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sim = np.dot(target_emb, db_emb) / (np.linalg.norm(target_emb) * np.linalg.norm(db_emb))
